In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import re

from setting_for_sdm.path_setting import path_list
from setting_for_sdm.date_setting import Date_Setting
from setting_for_sdm.constants import CONSTANTS

import lib.stats.stats as st
from lib.utils.file_io import *
from lib.utils.statistics import *
from lib.utils.settings import set_matplotlib
from matplotlib import pyplot as plt

import matplotlib as mpl
import lib.visualization.plot_generator as PlotGen
import lib.visualization.font_setting as font_setting
mpl.rcParams['font.family'] = font_setting.init_font()




Helvetica /home/mghan/.fonts/Helvetica/Helvetica Oblique.ttf
Registered font name: Helvetica


In [2]:
lang_list = list(CONSTANTS.src_extend.keys())

In [3]:
def draw_cognitive_complexity_scatter_plot(idx_num, viz_df, option_dict, output_dir, opt):
    sharey = False ## 또는 sharey=False
    sharex = True ## 또는 sharex=False

    fig, axs = mpl.pyplot.subplots(figsize = (12, 6), constrained_layout=True)

    # st_chow_1year = st.Stats(x, y,  0.95, viz_df[viz_df['rel_week'] ==0].index.values[0])
    # F_stat_1, p_value_1 = st_chow_1year.chow_test()
    plotgen = PlotGen.PlotGen()
    x= list(viz_df['rel_week'])
    y= list(viz_df['Cognitive Complexity'])
    
    plotgen.draw_regression_with_chow(axs
                                , f'Cognitive Complexity for {option_dict["selected_tags"]}({opt})' 
                                , x
                                , y
                                , None
                                , None
                                , None
                                , False
                                )

    fig.supxlabel("Weeks relative to ChatGPT release", fontsize=22) 
    fig.supylabel('Weekly average complexity', fontsize=22)
    filename = f"{idx_num}_scatter_plot_{option_dict['selected_tags']}_cognitive_complexity_{option_dict['year_range']}_{opt}.png"
    plt.savefig(
    os.path.join(output_dir, filename),
    dpi=300,
    bbox_inches='tight'
    )
    plt.close(fig)

In [4]:
def filter_df_with_tag(df, tags):
    pattern = '|'.join(re.escape(f'<{tag}>') for tag in tags)

    mask = df['tags'].astype(str).str.contains(pattern, regex=True)
    return df[mask]

In [7]:
run_id_start = 2000
for idx, lang in enumerate(lang_list[:10]):
    print(f'[visualizing....] start visualizing {lang} language')
    top_tags = load_json(f'/mnt/hdd/mghan/so_difficultyXavailability/data/{lang}_top_tags.json')
    bot_tags = load_json(f'/mnt/hdd/mghan/so_difficultyXavailability/data/{lang}_bot_tags.json')


    viz_dir = f'{path_list["data_root_dir"]}/result/code_complexity/run_id_{run_id_start+idx}'
    data_dir = f"{viz_dir}/data/csv"
    option_dict = load_json(f"{viz_dir}/data/option.json")
    output_dir = create_dir('./fig/')
    date_range = 'Weekly'
    std_date = Date_Setting[option_dict['year_range']]['std_date']

    df = pd.read_parquet(f'{option_dict["save_dir"]}/data/all_complexity.parquet')
    df['id'] = df['Path'].apply(lambda x : x.split('_')[1].split('.')[0])
    df[['id', 'Cognitive Complexity']] = df[['id', 'Cognitive Complexity']].astype(int)

    origin_df = load_df(option_dict['data_dir'], ['id', 'creationdate', 'title','tags', 'body'])
    filtered_top_df = filter_df_with_tag(origin_df, top_tags)
    filtered_bot_df = filter_df_with_tag(origin_df, bot_tags)


    top_viz_df = pd.merge(df, filtered_top_df, on = 'id')[['id', 'creationdate', 'Cognitive Complexity']]
    bot_viz_df = pd.merge(df, filtered_bot_df, on = 'id')[['id', 'creationdate', 'Cognitive Complexity']]

    top_viz_df['rel_week'] = np.floor((pd.to_datetime(top_viz_df['creationdate'], format='mixed')- std_date).dt.days/7)
    top_viz_df = (top_viz_df.groupby('rel_week', as_index=False)['Cognitive Complexity'].mean())

    bot_viz_df['rel_week'] = np.floor((pd.to_datetime(bot_viz_df['creationdate'], format='mixed')- std_date).dt.days/7)
    bot_viz_df = (bot_viz_df.groupby('rel_week', as_index=False)['Cognitive Complexity'].mean())
        
    draw_cognitive_complexity_scatter_plot(idx, top_viz_df, option_dict, output_dir, 'top 20%')
    draw_cognitive_complexity_scatter_plot(idx, bot_viz_df, option_dict, output_dir, 'bot 20%')
    print('[visualizing....] end visualizing {lang} language')
    


[visualizing....] start visualizing python language


100%|██████████| 65/65 [00:11<00:00,  5.64it/s]


[visualizing....] end visualizing {lang} language
[visualizing....] start visualizing javascript language


100%|██████████| 65/65 [00:09<00:00,  6.72it/s]


[visualizing....] end visualizing {lang} language
[visualizing....] start visualizing java language


...

[visualizing....] end visualizing {lang} language
[visualizing....] start visualizing swift language


100%|██████████| 65/65 [00:00<00:00, 69.08it/s]


[visualizing....] end visualizing {lang} language
[visualizing....] start visualizing kotlin language


100%|██████████| 65/65 [00:00<00:00, 77.15it/s]


[visualizing....] end visualizing {lang} language


In [ ]:
lang = 'javascript'
run_id_start = 3001
print(f'[visualizing....] start visualizing {lang} language')
top_tags = load_json(f'/mnt/hdd/mghan/so_difficultyXavailability/data/{lang}_top_tags.json')
bot_tags = load_json(f'/mnt/hdd/mghan/so_difficultyXavailability/data/{lang}_bot_tags.json')


viz_dir = f'{path_list["data_root_dir"]}/result/code_complexity/run_id_{run_id_start}'
data_dir = f"{viz_dir}/data/csv"
option_dict = load_json(f"{viz_dir}/data/option.json")
output_dir = create_dir('./fig/')
date_range = 'Weekly'
std_date = Date_Setting[option_dict['year_range']]['std_date']

df = pd.read_parquet(f'{option_dict["save_dir"]}/data/all_complexity.parquet')
df['id'] = df['Path'].apply(lambda x : x.split('_')[1].split('.')[0])
df[['id', 'Cognitive Complexity']] = df[['id', 'Cognitive Complexity']].astype(int)

origin_df = load_df(option_dict['data_dir'], ['id', 'creationdate', 'title','tags', 'body'])
filtered_top_df = filter_df_with_tag(origin_df, top_tags)
filtered_bot_df = filter_df_with_tag(origin_df, bot_tags)


top_viz_df = pd.merge(df, filtered_top_df, on = 'id')[['id', 'creationdate', 'Cognitive Complexity']]
bot_viz_df = pd.merge(df, filtered_bot_df, on = 'id')[['id', 'creationdate', 'Cognitive Complexity']]

top_viz_df['rel_week'] = np.floor((pd.to_datetime(top_viz_df['creationdate'], format='mixed')- std_date).dt.days/7)
top_viz_df = (top_viz_df.groupby('rel_week', as_index=False)['Cognitive Complexity'].mean())

bot_viz_df['rel_week'] = np.floor((pd.to_datetime(bot_viz_df['creationdate'], format='mixed')- std_date).dt.days/7)
bot_viz_df = (bot_viz_df.groupby('rel_week', as_index=False)['Cognitive Complexity'].mean())
    
draw_cognitive_complexity_scatter_plot(run_id_start, top_viz_df, option_dict, output_dir, 'top 20%')
draw_cognitive_complexity_scatter_plot(run_id_start, bot_viz_df, option_dict, output_dir, 'bot 20%')
print('[visualizing....] end visualizing {lang} language')

[visualizing....] start visualizing python language


100%|██████████| 65/65 [00:13<00:00,  4.93it/s]


[visualizing....] end visualizing {lang} language
